In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardt(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.29869934916496277
epoch:  1, loss: 0.2947741448879242
epoch:  2, loss: 0.29056665301322937
epoch:  3, loss: 0.2862968444824219
epoch:  4, loss: 0.28195419907569885
epoch:  5, loss: 0.2773298919200897
epoch:  6, loss: 0.272802472114563
epoch:  7, loss: 0.2682400047779083
epoch:  8, loss: 0.2636910378932953
epoch:  9, loss: 0.2592216730117798
epoch:  10, loss: 0.2547909617424011
epoch:  11, loss: 0.250379741191864
epoch:  12, loss: 0.24600043892860413
epoch:  13, loss: 0.24161197245121002
epoch:  14, loss: 0.2372579574584961
epoch:  15, loss: 0.23289044201374054
epoch:  16, loss: 0.2283971607685089
epoch:  17, loss: 0.2240002453327179
epoch:  18, loss: 0.2198188304901123
epoch:  19, loss: 0.21607959270477295
epoch:  20, loss: 0.21159638464450836
epoch:  21, loss: 0.20702365040779114
epoch:  22, loss: 0.20259229838848114
epoch:  23, loss: 0.19830390810966492
epoch:  24, loss: 0.19408969581127167
epoch:  25, loss: 0.19040291011333466
epoch:  26, loss: 0.18762828409671783

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.966890811920166
Test metrics:  R2 = 0.9320445656776428


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.26283541321754456
epoch:  1, loss: 0.10995602607727051
epoch:  2, loss: 0.08658323436975479
epoch:  3, loss: 0.062090348452329636
epoch:  4, loss: 0.042409323155879974
epoch:  5, loss: 0.03517643362283707
epoch:  6, loss: 0.03348808363080025
epoch:  7, loss: 0.032490309327840805
epoch:  8, loss: 0.03151506185531616
epoch:  9, loss: 0.03050904907286167
epoch:  10, loss: 0.02971469797194004
epoch:  11, loss: 0.029636749997735023
epoch:  12, loss: 0.025050105527043343
epoch:  13, loss: 0.0240484569221735
epoch:  14, loss: 0.02142215520143509
epoch:  15, loss: 0.017054883763194084
epoch:  16, loss: 0.015508294105529785
epoch:  17, loss: 0.012678329832851887
epoch:  18, loss: 0.010964025743305683
epoch:  19, loss: 0.010458807460963726
epoch:  20, loss: 0.01006191223859787
epoch:  21, loss: 0.009422162547707558
epoch:  22, loss: 0.008746487088501453
epoch:  23, loss: 0.008252497762441635
epoch:  24, loss: 0.007877499796450138
epoch:  25, loss: 0.007594384253025055
epoch:  

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9949055910110474
Test metrics:  R2 = 0.979946494102478
